In [ ]:
!pip install -q pyngrok bcrypt PyJWT streamlit streamlit-option-menu plotly
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

In [ ]:
%%writefile app.py
# from google.colab import userdata # Commented out as it's not needed in the app.py context
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
import streamlit.components.v1 as components
from email.utils import formatdate
from email.mime.text import MIMEText
import re # Import regular expression module

OTP_EXPIRY_MINUTES=5
SENDER_EMAIL = "yuvanesh1582005@gmail.com"
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD") # Get from environment variable
# 🚀 Force Streamlit Dark Theme — Nebula Control Tower console
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="dark"\nprimaryColor="#8B5CF6"\nbackgroundColor="#080B14"\nsecondaryBackgroundColor="#10162A"\ntextColor="#E8ECFB"\n')

st.set_page_config(page_title="Intelligent Freight Quote Generation", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

# ============================================================
# 🎨 DESIGN SYSTEM — "Nebula Control Tower"
# A freight-ops console reimagined as a night-sky command deck:
# deep space-navy canvas, a dual-tone signal (violet + cyan)
# standing in for two intersecting shipping routes, an animated
# particle-network layer suggesting a live logistics mesh, and
# monospace typography for anything data-like (codes, IDs, stats)
# — the way a dispatch terminal would render them.
# ============================================================
COLORS = {
    "bg_main": "#080B14",          # deep space-navy canvas
    "bg_sidebar": "#04060C",       # near-black terminal rail
    "bg_card": "#10162A",          # panel / manifest-card surface
    "bg_card_alt": "#0B1020",      # recessed surface (inputs, gauge bg)

    "text_main": "#C7CEE8",
    "text_heading": "#F3F5FC",
    "text_muted": "#7E88AC",

    "accent": "#8B5CF6",           # signal violet — primary route color
    "accent_soft": "rgba(139,92,246,0.14)",
    "accent_hover": "#7C3AED",
    "accent_text": "#F7F4FF",      # light ink text on violet buttons

    "accent2": "#22D3EE",          # signal cyan — secondary crossing route
    "accent2_soft": "rgba(34,211,238,0.14)",

    "amber": "#F5A524",            # tertiary status accent (used sparingly)
    "border": "#212A45",
    "border_light": "#1A2138",

    "success": "#34D399",
    "danger": "#F0546B"
}

JWT_SECRET = "super-secret-infosys-key-2026"

# Regex for email validation: At least 2 letters before @, 2 letters between @ and ., 2 letters after .
EMAIL_REGEX = r"^[a-zA-Z]{2,}@[a-zA-Z]{2,}\.[a-zA-Z]{2,}$"
# Regex for password complexity: Minimum 8 chars, at least one uppercase, one lowercase, one number, one special character
PASSWORD_REGEX = r"""^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[!@#$%^&*()_+={}\[\]:;\"'<>,.?/\\|`~-])[A-Za-z\d!@#$%^&*()_+={}\[\]:\"'<>?,./\\|`~-]{8,}$"""

# ============================================================
# 🎨 NEBULA CONTROL TOWER CSS
# Signature element: the dashed "route-divider" — a waypoint
# line under key headings, echoing a shipping route on a map,
# now rendered as a violet→cyan gradient dash.
# Only touches presentation (CSS + markup) — no logic below.
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500;600&display=swap');

    html, body, .stApp {{
        background:
            radial-gradient(circle at 10% -12%, rgba(139,92,246,0.14) 0%, transparent 42%),
            radial-gradient(circle at 96% 6%, rgba(34,211,238,0.10) 0%, transparent 38%),
            radial-gradient(circle at 50% 110%, rgba(139,92,246,0.06) 0%, transparent 45%),
            {COLORS['bg_main']} !important;
        font-family: 'Inter', sans-serif !important;
        color: {COLORS['text_main']} !important;
    }}

    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* --- Particle-network background layer: fixed, full-viewport, behind content --- */
    div[data-testid="stIframe"] {{
        position: fixed !important;
        top: 0 !important; left: 0 !important;
        width: 100vw !important; height: 100vh !important;
        z-index: 0 !important;
        pointer-events: none !important;
        border: none !important;
    }}
    div[data-testid="stIframe"] iframe {{
        width: 100vw !important; height: 100vh !important;
        border: none !important;
    }}

    /* Everything else sits above the particle layer */
    section[data-testid="stSidebar"], .block-container, header {{ position: relative; z-index: 1; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background: linear-gradient(135deg, {COLORS['accent']}, {COLORS['accent2']}) !important;
        border: none !important; border-radius: 10px !important; padding: 8px !important; margin: 10px !important;
        box-shadow: 0 0 0 1px rgba(139,92,246,0.3), 0 4px 14px rgba(34,211,238,0.25) !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['bg_main']} !important; color: {COLORS['bg_main']} !important; stroke: {COLORS['bg_main']} !important;
    }}

    .block-container {{ padding: 2.2rem 2.6rem !important; max-width: 1200px; animation: fadeInUp 0.4s ease both; }}
    @keyframes fadeInUp {{ from {{ opacity: 0; transform: translateY(6px); }} to {{ opacity: 1; transform: translateY(0); }} }}

    h1, h2, h3, h4 {{ font-family: 'Space Grotesk', sans-serif !important; color: {COLORS['text_heading']} !important; letter-spacing: -0.01em; }}
    label p {{ font-weight: 500 !important; color: {COLORS['text_muted']} !important; font-size: 12.5px !important; text-transform: uppercase; letter-spacing: 0.04em; }}

    /* --- Eyebrow / mono data labels — the "terminal readout" voice --- */
    .pn-eyebrow {{
        font-family: 'JetBrains Mono', monospace; font-size: 11.5px; font-weight: 500;
        letter-spacing: 0.16em; text-transform: uppercase; color: {COLORS['accent2']};
        display: flex; align-items: center; justify-content: center; gap: 8px;
    }}
    .pn-eyebrow::before, .pn-eyebrow::after {{ content: ""; width: 18px; height: 1px; background: {COLORS['border']}; }}
    .pn-mono {{ font-family: 'JetBrains Mono', monospace !important; }}

    /* --- Signature: route-divider (dual-tone dashed waypoint line) --- */
    .route-divider {{
        position: relative; width: 130px; height: 2px; margin: 16px auto 22px;
        background: repeating-linear-gradient(to right, {COLORS['accent']} 0px, {COLORS['accent']} 6px, transparent 6px, transparent 13px);
        opacity: 0.9;
    }}
    .route-divider .route-dot {{
        position: absolute; right: -3px; top: -3px; width: 8px; height: 8px; border-radius: 50%;
        background: {COLORS['accent2']}; box-shadow: 0 0 10px 2px rgba(34,211,238,0.75);
    }}

    /* --- Inputs — recessed terminal fields --- */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{
        background-color: {COLORS['bg_card_alt']} !important;
        border: 1.5px solid {COLORS['border']} !important;
        border-radius: 10px !important;
        transition: border-color 0.2s ease, box-shadow 0.2s ease !important;
    }}
    div[data-baseweb="input"]:focus-within, div[data-baseweb="select"]:focus-within {{
        border-color: {COLORS['accent2']} !important;
        box-shadow: 0 0 0 3px {COLORS['accent2_soft']} !important;
    }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_heading']} !important; -webkit-text-fill-color: {COLORS['text_heading']} !important; font-family: 'JetBrains Mono', monospace !important; font-size: 14px !important; }}

    /* --- Buttons — violet→cyan gradient signal, glow on hover --- */
    div[data-testid="stButton"] button {{
        background: linear-gradient(135deg, {COLORS['accent']} 0%, {COLORS['accent_hover']} 55%, {COLORS['accent2']} 130%) !important;
        color: {COLORS['accent_text']} !important;
        border: none !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 18px !important;
        box-shadow: 0 4px 14px rgba(139,92,246,0.22) !important;
        width: 100%; transition: transform 0.16s ease, box-shadow 0.16s ease, filter 0.16s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        filter: brightness(1.08) !important;
        transform: translateY(-2px) !important;
        box-shadow: 0 10px 26px rgba(34,211,238,0.28) !important;
    }}
    div[data-testid="stButton"] button:active {{ transform: translateY(0px) !important; }}

    /* --- Sidebar — the dispatch rail. Active item = dual-tone signal bar --- */
    section[data-testid="stSidebar"] {{
        background: {COLORS['bg_sidebar']} !important;
        border-right: 1px solid {COLORS['border_light']} !important;
    }}
    section[data-testid="stSidebar"] * {{ color: {COLORS['text_main']} !important; }}
    section[data-testid="stSidebar"] hr {{ border-color: {COLORS['border_light']} !important; }}

    /* --- Panels — manifest-card look: hairline top strip, violet→cyan gradient --- */
    .pn-card {{
        background: {COLORS['bg_card']};
        border: 1px solid {COLORS['border_light']};
        border-top: 2px solid transparent;
        border-image: linear-gradient(90deg, {COLORS['accent']}, {COLORS['accent2']}) 1;
        border-radius: 14px;
        padding: 26px;
        box-shadow: 0 10px 28px rgba(0,0,0,0.4);
        transition: transform 0.2s ease, box-shadow 0.2s ease;
    }}
    .pn-card:hover {{ transform: translateY(-2px); box-shadow: 0 16px 36px rgba(0,0,0,0.5); }}

    /* --- Alerts — recast Streamlit's default boxes as console readouts --- */
    div[data-testid="stAlert"], div[data-testid="stNotification"] {{
        background: {COLORS['bg_card_alt']} !important;
        border: 1px solid {COLORS['border']} !important;
        border-radius: 10px !important;
        color: {COLORS['text_main']} !important;
    }}

    .pn-stat:hover {{ transform: translateY(-3px); }}
    .pn-stat .pn-stat-val {{ font-family: 'JetBrains Mono', monospace; }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# 🌐 PARTICLE NETWORK BACKGROUND
# A quiet, drifting mesh of nodes and connecting threads —
# stands in for a live map of freight routes/hubs pulsing in
# the background. Fixed behind all content, non-interactive.
# ============================================================
def render_particle_network(accent="#8B5CF6", accent2="#22D3EE", n_particles=70):
    html_code = f"""
    <canvas id="pn-canvas" style="display:block;width:100vw;height:100vh;background:transparent;"></canvas>
    <script>
        const canvas = document.getElementById('pn-canvas');
        const ctx = canvas.getContext('2d');
        let W, H;
        function resize() {{
            W = canvas.width = window.innerWidth;
            H = canvas.height = window.innerHeight;
        }}
        window.addEventListener('resize', resize);
        resize();

        const COLOR_A = '{accent}';
        const COLOR_B = '{accent2}';
        const N = {n_particles};
        const LINK_DIST = 150;

        function hexToRgb(hex) {{
            const v = parseInt(hex.slice(1), 16);
            return [(v >> 16) & 255, (v >> 8) & 255, v & 255];
        }}
        const rgbA = hexToRgb(COLOR_A);
        const rgbB = hexToRgb(COLOR_B);

        const particles = [];
        for (let i = 0; i < N; i++) {{
            particles.push({{
                x: Math.random() * W,
                y: Math.random() * H,
                vx: (Math.random() - 0.5) * 0.35,
                vy: (Math.random() - 0.5) * 0.35,
                r: Math.random() * 1.6 + 0.8,
                c: Math.random() > 0.5 ? rgbA : rgbB
            }});
        }}

        function step() {{
            ctx.clearRect(0, 0, W, H);

            for (let i = 0; i < N; i++) {{
                const p = particles[i];
                p.x += p.vx;
                p.y += p.vy;
                if (p.x < 0 || p.x > W) p.vx *= -1;
                if (p.y < 0 || p.y > H) p.vy *= -1;

                for (let j = i + 1; j < N; j++) {{
                    const q = particles[j];
                    const dx = p.x - q.x, dy = p.y - q.y;
                    const dist = Math.sqrt(dx * dx + dy * dy);
                    if (dist < LINK_DIST) {{
                        const alpha = (1 - dist / LINK_DIST) * 0.16;
                        const mr = (p.c[0] + q.c[0]) / 2, mg = (p.c[1] + q.c[1]) / 2, mb = (p.c[2] + q.c[2]) / 2;
                        ctx.strokeStyle = `rgba(${{mr}},${{mg}},${{mb}},${{alpha}})`;
                        ctx.lineWidth = 1;
                        ctx.beginPath();
                        ctx.moveTo(p.x, p.y);
                        ctx.lineTo(q.x, q.y);
                        ctx.stroke();
                    }}
                }}
            }}
            for (let i = 0; i < N; i++) {{
                const p = particles[i];
                ctx.beginPath();
                ctx.arc(p.x, p.y, p.r, 0, Math.PI * 2);
                ctx.fillStyle = `rgba(${{p.c[0]}},${{p.c[1]}},${{p.c[2]}},0.65)`;
                ctx.fill();
            }}
            requestAnimationFrame(step);
        }}
        step();
    </script>
    """
    components.html(html_code, height=1, scrolling=False)

render_particle_network(accent=COLORS['accent'], accent2=COLORS['accent2'])

# ============================================================
# 🔷 LOGOMARK — "Hub & Route": a hexagonal node (network/hub)
# traced by a route line ending in a waypoint dot. Echoes the
# same route-divider language used elsewhere, so the icon isn't
# decoration — it's the visual thesis of the product, repeated.
# ============================================================
def logo_mark(size=40, ctx="a", badge=False):
    gid = f"ifqGrad_{ctx}"
    svg = f"""<svg width="{size}" height="{size}" viewBox="0 0 48 48" fill="none" xmlns="http://www.w3.org/2000/svg" style="display:block;">
        <defs>
            <linearGradient id="{gid}" x1="0" y1="0" x2="48" y2="48" gradientUnits="userSpaceOnUse">
                <stop stop-color="{COLORS['accent']}"/>
                <stop offset="1" stop-color="{COLORS['accent2']}"/>
            </linearGradient>
        </defs>
        <path d="M24 3L42 13.5V34.5L24 45L6 34.5V13.5L24 3Z" stroke="url(#{gid})" stroke-width="2.4"/>
        <path d="M13.5 29.5L21.5 19.5L27.5 25.5L34.5 14" stroke="url(#{gid})" stroke-width="2.8" stroke-linecap="round" stroke-linejoin="round" fill="none"/>
        <circle cx="34.5" cy="14" r="3.1" fill="url(#{gid})"/>
    </svg>"""
    if not badge:
        return svg
    pad = max(6, int(size * 0.28))
    return f"""<div style="display:inline-flex;padding:{pad}px;border-radius:16px;background:{COLORS['bg_card_alt']};border:1px solid {COLORS['border']};box-shadow:0 0 26px rgba(139,92,246,0.18);">{svg}</div>"""

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except:
        return None

def make_otp_token(email, otp):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp", "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp": return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]): return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError: return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception as e: return False, "Invalid or corrupted verification token."

def send_professional_email(to_email, otp, app_pass):
    # Plain text email
    msg = MIMEText(f"Your verification code for Intelligent Freight Quote Generation is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request this code, please ignore this email.")
    msg['From'] = f"Intelligent Freight Quote Generation Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Intelligent Freight Quote Generation - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Reply-To'] = SENDER_EMAIL

    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, app_pass if app_pass else "")
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e: return False, f"SMTP Error: {str(e)}"

for k, v in [
    ("token", None),
    ("page", "Login"),
    ("reset_email", None),
    ("reset_mode", None),
    ("otp_stage", "email"),
    ("otp_token", None),
    ("otp_verified", False)
]:
    if k not in st.session_state:
        st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Freight Quote Generation"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 0.4rem;">
        <div style="margin-bottom:12px;display:flex;justify-content:center;">{logo_mark(40, "auth", badge=True)}</div>
        <div class="pn-eyebrow">Secure Access Terminal</div>
        <h1 style="font-size:2rem !important;margin:10px 0 0;">Intelligent Freight Quote Generation</h1>
        <p style="color:{COLORS['text_muted']};font-size:13.5px;margin:4px 0 0;">{sub}</p>
    </div>
    <div class="route-divider"><span class="route-dot"></span></div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.05rem;font-weight:600;color:{COLORS['text_heading']};font-family:'Space Grotesk',sans-serif;">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            with st.container(border=False): # Add a container for better styling
                # st.markdown(f"<div class='pn-card'>", unsafe_allow_html=True)
                email = st.text_input("Email address", placeholder="you@example.com").lower().strip()
                pwd = st.text_input("Password", type="password", placeholder="••••••••").strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
                if col_l.button("Sign In →", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                    if r and check_txt(pwd, r[0]): st.session_state.token = make_jwt(email); navigate("Dashboard")
                    else: st.error("❌ Invalid credentials.")
                if col_c.button("Create Account", use_container_width=True): navigate("Signup")
                if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")
                st.markdown(f"</div>", unsafe_allow_html=True)

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Intelligent Freight Quote Generation today")
            with st.container(border=False): # Add a container for better styling
                # st.markdown(f"<div class='pn-card'>", unsafe_allow_html=True)
                uname = st.text_input("Full name / Username", placeholder="Jane Doe")
                email = st.text_input("Email address", placeholder="you@example.com").lower().strip()
                pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
                confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
                sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
                sa = st.text_input("Your answer", placeholder="Security answer")
                st.markdown("<br>", unsafe_allow_html=True)

                if st.button("Create Account & Login →", use_container_width=True):
                    if not uname or not email or not pwd or not sa:
                        st.error("⚠️ Please fill all fields.")
                    # elif not re.fullmatch(EMAIL_REGEX, email):
                    #     st.error("❌ Invalid email format. (e.g., ab@cd.ef)")
                    elif not re.fullmatch(PASSWORD_REGEX, pwd):
                        st.error("❌ Password must be at least 8 characters, with at least one uppercase, one lowercase, one number, and one special character.")
                    elif pwd != confirm_pwd:
                        st.error("❌ Passwords do not match.")
                    else:
                        try:
                            with get_db() as c:
                                c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                            st.session_state.token = make_jwt(email)
                            st.success("✅ Account created!")
                            time.sleep(1)
                            navigate("Dashboard")
                        except sqlite3.IntegrityError:
                            st.error("❌ Email or Username already registered.")

                st.markdown("<br>", unsafe_allow_html=True)
                if st.button("← Back to Sign In", use_container_width=True): navigate("Login")
                st.markdown(f"</div>", unsafe_allow_html=True)

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")
            if not st.session_state.reset_email:
                with st.container(border=False): # Add a container for better styling
                    # st.markdown(f"<div class='pn-card'>", unsafe_allow_html=True)
                    email = st.text_input("Registered email address", placeholder="you@example.com").lower().strip()
                    st.markdown("<br>", unsafe_allow_html=True)

                    col_sq, col_otp = st.columns(2)
                    if col_sq.button("Via Security Question", use_container_width=True):
                        # if not re.fullmatch(EMAIL_REGEX, email):
                        #     st.error("❌ Invalid email format. (e.g., ab@cd.ef)")
                        # else:
                            with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                            if r:
                                st.session_state.reset_email = email
                                st.session_state.sq_p = r[0]
                                st.session_state.reset_mode = "sq"
                                st.rerun()
                            else: st.error("❌ Email not found.")

                    if col_otp.button("Via OTP", use_container_width=True):
                        # if not re.fullmatch(EMAIL_REGEX, email):
                        #     st.error("❌ Invalid email format. (e.g., ab@cd.ef)")
                        # else:
                            with get_db() as c:
                                r = c.execute(
                                    "SELECT 1 FROM users WHERE email=?",
                                    (email,)
                                ).fetchone()

                            if r:
                                otp = generate_otp()

                                ok, msg = send_professional_email(
                                    email,
                                    otp,
                                    EMAIL_PASSWORD
                                )

                                if ok:
                                    st.session_state.reset_email = email
                                    st.session_state.otp_token = make_otp_token(email, otp)
                                    st.session_state.reset_mode = "otp"

                                    st.success("✅ OTP sent successfully. Check your email.")
                                    time.sleep(1)
                                    st.rerun()

                                else:
                                    st.error(f"❌ {msg}")

                            else:
                                st.error("❌ Email not found.")
                    st.markdown(f"</div>", unsafe_allow_html=True)

            else:
                if st.session_state.get("reset_mode") == "sq":
                    with st.container(border=False): # Add a container for better styling
                        # st.markdown(f"<div class='pn-card'>", unsafe_allow_html=True)
                        st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                        ans = st.text_input("Your answer").lower().strip()
                        npw = st.text_input("New password (min 8 chars)", type="password").strip()
                        confirm_npw = st.text_input("Confirm new password", type="password").strip()
                        st.markdown("<br>", unsafe_allow_html=True)
                        if st.button("Reset Password →", use_container_width=True):
                            if not re.fullmatch(PASSWORD_REGEX, npw):
                                st.error("❌ Password must be at least 8 characters, with at least one uppercase, one lowercase, one number, and one special character.")
                            elif npw != confirm_npw:
                                st.error("❌ Passwords do not match.")
                            else:
                                with get_db() as c:
                                    user_data = c.execute("SELECT password_hash, security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                                if user_data and check_txt(ans, user_data[1]):
                                    old_password_hash = user_data[0]
                                    if check_txt(npw, old_password_hash):
                                        st.error("❌ New password cannot be the same as the old password.")
                                    else:
                                        with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                        st.success("✅ Password updated successfully!"); time.sleep(1); st.session_state.reset_email = None; navigate("Login")
                                else: st.error("❌ Incorrect security answer.")
                        st.markdown(f"</div>", unsafe_allow_html=True)

                elif st.session_state.get("reset_mode") == "otp":
                    with st.container(border=False): # Add a container for better styling
                        # st.markdown(f"<div class='pn-card'>", unsafe_allow_html=True)
                        st.info(f"📧 OTP sent to **{st.session_state.reset_email}**")

                        if not st.session_state.otp_verified:
                            otp_input = st.text_input("Enter 6-digit OTP", max_chars=6)
                            if st.button("Verify OTP", use_container_width=True):
                                ok, msg = verify_otp_token(
                                    st.session_state.otp_token,
                                    otp_input,
                                    st.session_state.reset_email
                                )
                                if ok:
                                    st.success("✅ OTP verified successfully! Now set your new password.")
                                    st.session_state.otp_verified = True
                                    st.rerun()
                                else:
                                    st.error(msg)
                        else: # otp_verified is True
                            npw = st.text_input(
                                "New Password (min 8 chars)",
                                type="password"
                            ).strip()

                            confirm_npw = st.text_input(
                                "Confirm New Password",
                                type="password"
                            ).strip()

                            if st.button("Set New Password →", use_container_width=True):
                                if not re.fullmatch(PASSWORD_REGEX, npw):
                                    st.error("❌ Password must be at least 8 characters, with at least one uppercase, one lowercase, one number, and one special character.")
                                elif npw != confirm_npw:
                                    st.error("❌ Passwords do not match.")
                                else:
                                    with get_db() as c:
                                        old_password_hash_row = c.execute("SELECT password_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                                        old_password_hash = old_password_hash_row[0] if old_password_hash_row else None

                                    if old_password_hash and check_txt(npw, old_password_hash):
                                        st.error("❌ New password cannot be the same as the old password.")
                                    else:
                                        with get_db() as c:
                                            c.execute(
                                                "UPDATE users SET password_hash=? WHERE email=?",
                                                (
                                                    hash_txt(npw),
                                                    st.session_state.reset_email
                                                )
                                            )
                                        st.success("✅ Password updated successfully!")
                                        time.sleep(1)
                                        st.session_state.reset_email = None
                                        st.session_state.reset_mode = None
                                        st.session_state.otp_token = None
                                        st.session_state.otp_verified = False # Reset for next time
                                        navigate("Login")
                        st.markdown(f"</div>", unsafe_allow_html=True)

                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("← Cancel", use_container_width=True):
                        st.session_state.reset_email = None
                        st.session_state.reset_mode = None
                        st.session_state.otp_token = None
                        st.session_state.otp_verified = False
                        navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    # 🚀 SIDEBAR MENU — dispatch rail
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:22px 8px 14px;text-align:center;">
            <div style="display:flex;justify-content:center;margin-bottom:8px;">{logo_mark(26, "sidebar", badge=True)}</div>
            <div style="font-weight:700;font-size:15px;color:{COLORS['text_heading']};font-family:'Space Grotesk',sans-serif;">Intelligent Freight</div>
            <div class="pn-mono" style="font-size:10.5px;color:{COLORS['accent2']};letter-spacing:0.12em;margin-top:2px;">{"ADMIN · CONTROL" if email=="infosys@ai" else "OPS · TERMINAL"}</div>
        </div><hr style="border-color:{COLORS['border_light']};margin:4px 0 12px;">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={
                               "container": {"background-color": "transparent", "padding": "0px"},
                               "icon": {"color": COLORS['text_muted'], "font-size": "15px"},
                               "nav-link": {"color": COLORS['text_main'], "font-weight": "500", "border-radius": "6px", "margin": "3px 0", "border-left": "3px solid transparent"},
                               "nav-link-selected": {"background-color": COLORS['bg_card'], "color": COLORS['accent2'], "border-left": f"3px solid {COLORS['accent2']}", "font-weight": "600"}
                           })
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['bg_card']};border:1px solid {COLORS['border_light']};border-left:3px solid {COLORS['accent']};border-radius:14px;padding:22px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div style="display:flex;align-items:center;gap:16px;">
                {logo_mark(30, "bannerAdmin", badge=True)}
                <div>
                    <div class="pn-mono" style="font-size:10.5px;color:{COLORS['accent2']};letter-spacing:0.14em;margin-bottom:4px;">ADMIN CONTROL PANEL</div>
                    <h1 style="color:{COLORS['text_heading']} !important;margin:0;font-size:22px !important;">Intelligent Freight Quote Generation</h1>
                </div>
            </div>
            <div class="pn-mono" style="background:{COLORS['bg_card_alt']};border:1px solid {COLORS['border']};padding:8px 18px;border-radius:8px;font-weight:600;color:{COLORS['accent2']};">🛒 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        with st.container(border=False):
            st.markdown(f"<div class='pn-card' style='text-align:center;padding:60px 20px;'>", unsafe_allow_html=True)
            st.markdown(f"""
            <h1 style="font-size:34px !important;margin-bottom:10px;">🛒 Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:15px;font-weight:500;">Welcome to the Administrator area.</p>
            """, unsafe_allow_html=True)
            st.markdown(f"</div>", unsafe_allow_html=True)

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['bg_card']};border:1px solid {COLORS['border_light']};border-left:3px solid {COLORS['accent']};border-radius:14px;padding:22px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div style="display:flex;align-items:center;gap:16px;">
                {logo_mark(30, "bannerUser", badge=True)}
                <div>
                    <div class="pn-mono" style="font-size:10.5px;color:{COLORS['accent2']};letter-spacing:0.14em;margin-bottom:4px;">OPERATIONS DASHBOARD</div>
                    <h1 style="color:{COLORS['text_heading']} !important;margin:0;font-size:22px !important;">Intelligent Freight Quote Generation</h1>
                </div>
            </div>
            <div class="pn-mono" style="background:{COLORS['bg_card_alt']};border:1px solid {COLORS['border']};padding:8px 18px;border-radius:8px;font-weight:600;color:{COLORS['accent2']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)
        st.markdown(f"""
        <h2 style="font-size:1.9rem !important; margin-top: 0.5rem; color:{COLORS['text_heading']} !important;">Welcome, {uname}!</h2>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛒", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card pn-stat" style="text-align:center;">
                <div style="font-size:26px;">{icon}</div>
                <div class="pn-stat-val" style="font-size:24px;font-weight:600;color:{COLORS['accent2']};margin-top:4px;">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:11.5px;font-weight:500;margin-top:4px;text-transform:uppercase;letter-spacing:0.05em;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent2']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
EMAIL_PASSWORD = userdata.get('EMAIL_PASSWORD') # Retrieve EMAIL_PASSWORD here
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
# Pass EMAIL_PASSWORD as an environment variable
my_env = os.environ.copy()
my_env["EMAIL_PASSWORD"] = EMAIL_PASSWORD

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=my_env
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Intelligent Freight Quote Generation Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")

🚀 Intelligent Freight Quote Generation Live URL: https://blurry-identical-strung.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.
